In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time


In [2]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: mps


In [3]:
BATCH_SIZE = 32
BLOCK_SIZE = 512     # context length
EMBED_DIM = 512
NUM_HEADS = 16
NUM_LAYERS = 6
DROPOUT = 0.2
LR = 3e-4
EPOCHS = 25


In [4]:
import torch

files = [
    "/Users/abhin-zstch1563/Documents/AI/DL/RNN/poem/ponniayan.txt"
]

text = ""
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        text += f.read() + "\n"


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

def decode(t):
    return "".join([itos[i] for i in t])

data = encode(text)


In [5]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("Vocab size:", vocab_size)
print("Train tokens:", len(train_data))


Vocab size: 93
Train tokens: 3363011


In [6]:
chars

['\n',
 ' ',
 '!',
 '"',
 '#',
 '&',
 "'",
 '(',
 ')',
 '*',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '<',
 '>',
 '?',
 'L',
 '_',
 '`',
 'd',
 'e',
 'h',
 'i',
 'p',
 'r',
 'v',
 'அ',
 'ஆ',
 'இ',
 'ஈ',
 'உ',
 'ஊ',
 'எ',
 'ஏ',
 'ஐ',
 'ஒ',
 'ஓ',
 'ஔ',
 'க',
 'ங',
 'ச',
 'ஜ',
 'ஞ',
 'ட',
 '\u0ba2',
 'ண',
 'த',
 'ந',
 'ன',
 'ப',
 'ம',
 'ய',
 'ர',
 'ற',
 'ல',
 'ள',
 'ழ',
 'வ',
 'ஷ',
 'ஸ',
 'ஹ',
 'ா',
 'ி',
 'ீ',
 'ு',
 'ூ',
 'ெ',
 'ே',
 'ை',
 'ொ',
 'ோ',
 'ௌ',
 '்',
 '–',
 '—',
 '‘',
 '’',
 '“',
 '”',
 '…']

In [7]:
def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(0, len(src) - BLOCK_SIZE, (BATCH_SIZE,))
    
    x = torch.stack([src[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([src[i+1:i+BLOCK_SIZE+1] for i in ix])
    
    return x.to(DEVICE), y.to(DEVICE)


In [8]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out = nn.Linear(embed_dim, embed_dim)

        self.register_buffer(
            "mask", torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)


In [9]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = SelfAttention(EMBED_DIM, NUM_HEADS)
        self.ff = nn.Sequential(
            nn.Linear(EMBED_DIM, 4 * EMBED_DIM),
            nn.ReLU(),
            nn.Linear(4 * EMBED_DIM, EMBED_DIM)
        )
        self.ln1 = nn.LayerNorm(EMBED_DIM)
        self.ln2 = nn.LayerNorm(EMBED_DIM)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [10]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, EMBED_DIM)
        self.pos_emb = nn.Embedding(BLOCK_SIZE, EMBED_DIM)

        self.blocks = nn.Sequential(
            *[TransformerBlock() for _ in range(NUM_LAYERS)]
        )

        self.ln = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=DEVICE))
        x = tok + pos

        x = self.blocks(x)
        x = self.ln(x)
        logits = self.head(x)

        if targets is None:
            return logits

        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            targets.view(-1)
        )
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature= 1e-8):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            # logits = logits / temperature
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx


In [11]:
model = Transformer().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

sum(p.numel() for p in model.parameters()) / 1e6


19.272797

In [12]:
for epoch in range(EPOCHS):
    model.train()
    start = time.time()

    for step in range(200):
        x, y = get_batch("train")
        _, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Time: {time.time()-start:.1f}s")


Epoch 1 | Loss: 2.2711 | Time: 167.3s
Epoch 2 | Loss: 2.0756 | Time: 168.3s
Epoch 3 | Loss: 1.5828 | Time: 169.1s
Epoch 4 | Loss: 1.3688 | Time: 171.4s
Epoch 5 | Loss: 1.2787 | Time: 173.4s
Epoch 6 | Loss: 1.2140 | Time: 175.4s
Epoch 7 | Loss: 1.1370 | Time: 176.6s
Epoch 8 | Loss: 1.0992 | Time: 173.5s
Epoch 9 | Loss: 1.0551 | Time: 173.8s
Epoch 10 | Loss: 1.0144 | Time: 174.4s
Epoch 11 | Loss: 1.0111 | Time: 174.5s
Epoch 12 | Loss: 1.0274 | Time: 173.9s
Epoch 13 | Loss: 0.9556 | Time: 174.0s
Epoch 14 | Loss: 0.9285 | Time: 173.5s
Epoch 15 | Loss: 0.9208 | Time: 175.5s
Epoch 16 | Loss: 0.9312 | Time: 175.6s
Epoch 17 | Loss: 0.8764 | Time: 178.7s
Epoch 18 | Loss: 0.8653 | Time: 178.5s
Epoch 19 | Loss: 0.8348 | Time: 177.1s
Epoch 20 | Loss: 0.8271 | Time: 169.8s
Epoch 21 | Loss: 0.8287 | Time: 170.3s
Epoch 22 | Loss: 0.8200 | Time: 172.5s
Epoch 23 | Loss: 0.7768 | Time: 169.6s
Epoch 24 | Loss: 0.7908 | Time: 171.9s
Epoch 25 | Loss: 0.7713 | Time: 171.7s


In [13]:
model.eval()

context = torch.zeros((1, 1), dtype=torch.long).to(DEVICE)
generated = model.generate(context, max_new_tokens=500)

print(decode(generated[0].tolist()))



“சொன்னதைப் பொறுக்க என்ன விரும்புகிறேன்?” என்று திருமலையப்பன் கேட்டான்.
“இது என்ன வேடிக்கை? அந்தரங்கமானவன் ஓடி வர ஆவுகிறதா?”
இளவரசர் ஆதித்த கரிகாலரை மீறிக் கொண்டார்.
“உண்மையைச் சொல்ல வேண்டுமா; எதற்காக?”
“நீக்கண்ணா! என்னைப்பற்றி வந்தியத்தேவனை அவன் மூலமகாமவர் கூற வார்த்தைகளின் தலைவர் பருக ஆள் என்ன நேர்ந்து விட்டது? அவை அம்மாவே பிரியப்படுகிறான்.
வந்தியத்தேவன் எத்தனை காலமாகவ கைவன் மிகவும் ஆர்வமாகப் பொய்யாகக் கத்துக் கொண்டிருந்தான். அந்தக் கைகள் கைகளிலும் ஒரு கை அகர்ந்து கொண்டு தம் உட்புத் துணியினால் 


In [14]:
for _ in range(3):
    out = model.generate(
        torch.zeros((1,1), dtype=torch.long).to(DEVICE),
        max_new_tokens=300
    )
    print(decode(out[0].tolist()))
    print("="*80)




அத்தியாயம் 12 - பிறை ஒன்றும் சொல்லித்தானே! அதற்காக “அம்மா, என்ன? நீ ஒப்புக்கொள்ள வேண்டாம்!” என்று தேவராளன் ஆபரணம் இருவர் இச்சமயம் தொடங்கினான்.
“உண்மையானால் இளவரசர் சேர்த்துக் கொண்டு போ! அந்த வீரப் புரடல் எதற்காகக் கிடைக்கிறார்?”
“எனக்கு எச்சரிக்கை செய்திருக்கிறாய். வருகிறவரை கடலில் போகிறேன்.”
“இருந

நாராயணா! உன்னைப் போல் யாரேனும் கொல்லிற்று”
“வேறு எப்படி நாடெங்கும் அப்படித்தான் எறிந்தேன்? எப்போது?”
“தாங்கள் கேட்கச் சொல்கிறேன்! விஷத்தைப்பற்றி நாங்கள் எங்கேயோ கேட்கிறோம்.”
“பெண்ணே! இளவரசே! எத்தனையோ ஓர் உறைத்துக் கொண்டிருக்கிறேன். ஆமாதிரி இந்தக் குற்றம் சொல்கிறீர்களே! உண்மையானதற்காக இந்த மரத்தை எனக

“தேவி! இடந்தான் ஏன் சொன்னேனே? இன்று ஆண்டுகளுக்கு முக்கியம் என்ன செய்து பார்க்கிறேன்!” என்று வந்தியத்தேவன் சிரித்தான்.
மதுராந்தகத் தேவரின் பூங்குழலி ஜயகோஷம் என்று இளவரசர் கம்பீரமான திரும்பிப் பார்த்தான். தேவரின் வாழ்க்கை வந்தியத்தேவனை ரொம்பக் கயிறுகிறது. ஆனால் கந்தமாறன் அங்கே இருந்திருக்கிறான், அவனுட


In [15]:
torch.save(model.state_dict(), "PS_weights.pt")

In [ ]:


# அத்தியாயம் 12 - பிறை ஒன்றும் சொல்லித்தானே! அதற்காக “அம்மா, என்ன? நீ ஒப்புக்கொள்ள வேண்டாம்!” என்று தேவராளன் ஆபரணம் இருவர் இச்சமயம் தொடங்கினான்.
# “உண்மையானால் இளவரசர் சேர்த்துக் கொண்டு போ! அந்த வீரப் புரடல் எதற்காகக் கிடைக்கிறார்?”
# “எனக்கு எச்சரிக்கை செய்திருக்கிறாய். வருகிறவரை கடலில் போகிறேன்.”
# “இருந
# ================================================================================

# நாராயணா! உன்னைப் போல் யாரேனும் கொல்லிற்று”
# “வேறு எப்படி நாடெங்கும் அப்படித்தான் எறிந்தேன்? எப்போது?”
# “தாங்கள் கேட்கச் சொல்கிறேன்! விஷத்தைப்பற்றி நாங்கள் எங்கேயோ கேட்கிறோம்.”
# “பெண்ணே! இளவரசே! எத்தனையோ ஓர் உறைத்துக் கொண்டிருக்கிறேன். ஆமாதிரி இந்தக் குற்றம் சொல்கிறீர்களே! உண்மையானதற்காக இந்த மரத்தை எனக
# ================================================================================

# “தேவி! இடந்தான் ஏன் சொன்னேனே? இன்று ஆண்டுகளுக்கு முக்கியம் என்ன செய்து பார்க்கிறேன்!” என்று வந்தியத்தேவன் சிரித்தான்.
# மதுராந்தகத் தேவரின் பூங்குழலி ஜயகோஷம் என்று இளவரசர் கம்பீரமான திரும்பிப் பார்த்தான். தேவரின் வாழ்க்கை வந்தியத்தேவனை ரொம்பக் கயிறுகிறது. ஆனால் கந்தமாறன் அங்கே இருந்திருக்கிறான், அவனுட
# ================================================================================